In [1]:
import pandas as pd

school_df = pd.read_json('./data/raw_json/rdb_schools.json')
school_df.head()
school_df = school_df.rename(columns = {'address': 'location'})

In [2]:
faq_df = pd.read_json('./data/raw_json/rdb_faqs.json')
faq_df.head()

,school_code,category,question,answer,faq_id
0,NYU,Admissions,What are the key deadlines for undergraduate a...,"For undergraduate admissions at NYU, the key d...",NYU_FAQ_1
1,NYU,Admissions,Is NYU test-optional?,"Yes, NYU is test-optional through the 2026-202...",NYU_FAQ_2
2,NYU,Financial Aid,Does NYU meet 100% of demonstrated financial n...,"Yes, NYU will meet 100% of demonstrated financ...",NYU_FAQ_3
3,NYU,International Applicants,What English proficiency exams does NYU accept...,Non-native speakers may need to submit scores ...,NYU_FAQ_4
4,NYU,Spring Admissions,What is the benefit of Spring Admission at NYU?,Spring Admission allows students to start thei...,NYU_FAQ_5


In [3]:
# !pip install mysql-connector-python

In [4]:
import mysql.connector

# -------------SQL 연결-----------------#
connection = mysql.connector.connect(
    host = 'localhost', # MySQL 서버 주소(iP)
    user = 'ohgiraffers', # 사용자 이름
    password = 'ohgiraffers', # 비밀번호
    database = 'universirydb' #사용할 DB 스키마
)
cursor = connection.cursor(buffered=True)
# -------------------------------------#
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

('universirydb',)


In [5]:
faq_df = pd.merge(school_df,faq_df, on = 'school_code', how = 'inner')
faq_df


,school_code,name,country,location,category,question,answer,faq_id
0,NYU,New York University,USA,"New York, NY",Admissions,What are the key deadlines for undergraduate a...,"For undergraduate admissions at NYU, the key d...",NYU_FAQ_1
1,NYU,New York University,USA,"New York, NY",Admissions,Is NYU test-optional?,"Yes, NYU is test-optional through the 2026-202...",NYU_FAQ_2
2,NYU,New York University,USA,"New York, NY",Financial Aid,Does NYU meet 100% of demonstrated financial n...,"Yes, NYU will meet 100% of demonstrated financ...",NYU_FAQ_3
3,NYU,New York University,USA,"New York, NY",International Applicants,What English proficiency exams does NYU accept...,Non-native speakers may need to submit scores ...,NYU_FAQ_4
4,NYU,New York University,USA,"New York, NY",Spring Admissions,What is the benefit of Spring Admission at NYU?,Spring Admission allows students to start thei...,NYU_FAQ_5
5,USC,University of Southern California,United States,"Los Angeles, California",Dates and Deadlines,What are the important application deadlines f...,USC has the following key deadlines: Early Act...,USC_FAQ_1
6,USC,University of Southern California,United States,"Los Angeles, California",Admission Requirements,What is included in the application checklist ...,The application checklist for USC includes: Th...,USC_FAQ_2
7,USC,University of Southern California,United States,"Los Angeles, California",Application Process,Does USC accept home-schooled students differe...,"Yes, home-schooled students applying to USC mu...",USC_FAQ_3
8,USC,University of Southern California,United States,"Los Angeles, California",Financial Aid,Does USC offer need-based financial aid to int...,"No, USC does not provide need-based financial ...",USC_FAQ_4
9,USC,University of Southern California,United States,"Los Angeles, California",International Students,What are the English proficiency requirements ...,International applicants whose native language...,USC_FAQ_5


In [ ]:
# 2. 이제 다시 실행 (주의: cursor = cursor.execute() 라고 쓰면 안 됩니다!)
cursor.execute('SHOW TABLES')

# 3. 결과 출력
for table in cursor:
    print(table)
school_df_sql = """INSERT INTO school_info(school_name, country , location)  
                    VALUE (%s, %s, %s)"""
# for _, row in school_df.iterrows(): # 데이터프레임을 한줄씩 쓰는것
#     cursor.execute(school_df_sql, (
#         row['name'],
#         row['country'],
#         row['location']))
    
school_data = school_df[['name', 'country', 'location']].values.tolist()
cursor.executemany(school_df_sql, school_data)
connection.commit()

print("school 완료")

cursor.execute('SELECT * FROM school_info')
rows = cursor.fetchall()
school_id_map = {row[1]: row[0] for row in rows}
faq_df['school_id'] = faq_df['name'].map(school_id_map) # 이름으로 primary key 추가
faq_df.head()

('faq_info',)
('school_info',)
school 완료


,school_code,name,country,location,category,question,answer,faq_id,school_id
0,NYU,New York University,USA,"New York, NY",Admissions,What are the key deadlines for undergraduate a...,"For undergraduate admissions at NYU, the key d...",NYU_FAQ_1,101
1,NYU,New York University,USA,"New York, NY",Admissions,Is NYU test-optional?,"Yes, NYU is test-optional through the 2026-202...",NYU_FAQ_2,101
2,NYU,New York University,USA,"New York, NY",Financial Aid,Does NYU meet 100% of demonstrated financial n...,"Yes, NYU will meet 100% of demonstrated financ...",NYU_FAQ_3,101
3,NYU,New York University,USA,"New York, NY",International Applicants,What English proficiency exams does NYU accept...,Non-native speakers may need to submit scores ...,NYU_FAQ_4,101
4,NYU,New York University,USA,"New York, NY",Spring Admissions,What is the benefit of Spring Admission at NYU?,Spring Admission allows students to start thei...,NYU_FAQ_5,101
5,USC,University of Southern California,United States,"Los Angeles, California",Dates and Deadlines,What are the important application deadlines f...,USC has the following key deadlines: Early Act...,USC_FAQ_1,102
6,USC,University of Southern California,United States,"Los Angeles, California",Admission Requirements,What is included in the application checklist ...,The application checklist for USC includes: Th...,USC_FAQ_2,102
7,USC,University of Southern California,United States,"Los Angeles, California",Application Process,Does USC accept home-schooled students differe...,"Yes, home-schooled students applying to USC mu...",USC_FAQ_3,102
8,USC,University of Southern California,United States,"Los Angeles, California",Financial Aid,Does USC offer need-based financial aid to int...,"No, USC does not provide need-based financial ...",USC_FAQ_4,102
9,USC,University of Southern California,United States,"Los Angeles, California",International Students,What are the English proficiency requirements ...,International applicants whose native language...,USC_FAQ_5,102


In [9]:
faq_df_sql = """INSERT INTO faq_info(school_id, question , answer, category)  
                    VALUE (%s, %s, %s, %s)"""

cols = ['school_id', 'question', 'answer', 'category']

faq_data = faq_df[cols] \
    .astype(object) \
    .where(pd.notnull(faq_df[cols]), None) \
    .values.tolist()

# 6️⃣ INSERT
cursor.executemany(faq_df_sql, faq_data)
connection.commit()

print("완료")

완료
